# AI Governance Risk Assessment — Visualization
**FairFacts Consulting LLC** | Chiemeka Okoronkwo  
Framework: NIST AI RMF · ISO/IEC 42001 · EU AI Act · GDPR/CCPA

This notebook visualizes the AI governance risk register using a 5×5 likelihood-impact heat map, risk level distribution, and domain breakdown.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from matplotlib.colors import ListedColormap
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 11
print('Libraries loaded.')

In [ ]:
# Load the risk register
df = pd.read_csv('ai_risk_register.csv')

# Verify risk level calculation
def score_to_level(score):
    if score >= 15: return 'Critical'
    elif score >= 8: return 'High'
    elif score >= 4: return 'Medium'
    else: return 'Low'

df['Calculated Level'] = df['Risk Score'].apply(score_to_level)

print(f'Total risks loaded: {len(df)}')
print('\nRisk level distribution:')
print(df['Risk Level'].value_counts())
df[['Risk ID', 'Risk Description', 'Domain', 'Likelihood (1-5)', 'Impact (1-5)', 'Risk Score', 'Risk Level', 'Status']].head(10)

In [ ]:
# ── 5×5 Risk Heat Map ──────────────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(10, 7))

# Color mapping per cell score
def cell_color(score):
    if score >= 15: return '#F7C1C1'   # Critical — red
    elif score >= 8: return '#FAC775'  # High — amber
    elif score >= 4: return '#C0DD97'  # Medium — green
    else: return '#B5D4F4'             # Low — blue

# Draw grid
for l in range(1, 6):      # likelihood rows
    for i in range(1, 6):  # impact cols
        score = l * i
        color = cell_color(score)
        rect = mpatches.FancyBboxPatch(
            (i - 0.5, l - 0.5), 1, 1,
            boxstyle='square,pad=0',
            facecolor=color, edgecolor='white', linewidth=1.5
        )
        ax.add_patch(rect)
        ax.text(i, l, str(score), ha='center', va='center',
                fontsize=13, fontweight='bold', color='#333333')

# Plot risk dots
for _, row in df.iterrows():
    l = row['Likelihood (1-5)']
    i = row['Impact (1-5)']
    ax.plot(i, l, 'o', color='#2C2C2A', markersize=6, zorder=5, alpha=0.7)
    ax.annotate(row['Risk ID'], (i, l),
                textcoords='offset points', xytext=(6, 4),
                fontsize=7.5, color='#2C2C2A')

# Axes formatting
ax.set_xlim(0.5, 5.5)
ax.set_ylim(0.5, 5.5)
ax.set_xticks(range(1, 6))
ax.set_yticks(range(1, 6))
ax.set_xticklabels(['1\nNegligible', '2\nMinor', '3\nModerate', '4\nMajor', '5\nSevere'], fontsize=10)
ax.set_yticklabels(['1\nRare', '2\nUnlikely', '3\nPossible', '4\nLikely', '5\nAlmost\nCertain'], fontsize=10)
ax.set_xlabel('Impact →', fontsize=12, labelpad=10)
ax.set_ylabel('← Likelihood', fontsize=12, labelpad=10)
ax.set_title('AI Governance Risk Matrix — FairFacts Consulting', fontsize=14, fontweight='bold', pad=15)

# Legend
legend_patches = [
    mpatches.Patch(color='#F7C1C1', label='Critical (15–25)'),
    mpatches.Patch(color='#FAC775', label='High (8–12)'),
    mpatches.Patch(color='#C0DD97', label='Medium (4–6)'),
    mpatches.Patch(color='#B5D4F4', label='Low (1–3)'),
]
ax.legend(handles=legend_patches, loc='upper left', bbox_to_anchor=(1.01, 1),
          frameon=True, fontsize=10, title='Risk Level', title_fontsize=10)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('risk_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Heat map saved as risk_matrix.png')

In [ ]:
# ── Risk Level Distribution (Bar Chart) ────────────────────────────────────

level_order = ['Critical', 'High', 'Medium', 'Low']
level_colors = {'Critical': '#E24B4A', 'High': '#EF9F27', 'Medium': '#639922', 'Low': '#378ADD'}
counts = df['Risk Level'].value_counts().reindex(level_order, fill_value=0)

fig, ax = plt.subplots(figsize=(8, 4.5))
bars = ax.bar(counts.index, counts.values,
               color=[level_colors[l] for l in counts.index],
               edgecolor='white', linewidth=1, width=0.55)

for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.1,
            str(val), ha='center', va='bottom', fontsize=12, fontweight='bold')

ax.set_title('Risk Level Distribution', fontsize=13, fontweight='bold', pad=12)
ax.set_ylabel('Number of Risks', fontsize=11)
ax.set_ylim(0, counts.max() + 2)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_color('#CCCCCC')
ax.spines['bottom'].set_color('#CCCCCC')
ax.yaxis.set_tick_params(labelsize=10)
plt.tight_layout()
plt.savefig('risk_level_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Bar chart saved as risk_level_distribution.png')

In [ ]:
# ── Risks by Domain (Horizontal Bar) ────────────────────────────────────────

domain_counts = df.groupby('Domain').size().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 4.5))
bars = ax.barh(domain_counts.index, domain_counts.values,
               color='#AFA9EC', edgecolor='white', linewidth=1, height=0.55)

for bar, val in zip(bars, domain_counts.values):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height() / 2,
            str(val), va='center', fontsize=11, fontweight='bold')

ax.set_title('Number of Risks by Domain', fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Number of Risks', fontsize=11)
ax.set_xlim(0, domain_counts.max() + 1.5)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_color('#CCCCCC')
ax.spines['bottom'].set_color('#CCCCCC')
plt.tight_layout()
plt.savefig('risks_by_domain.png', dpi=150, bbox_inches='tight')
plt.show()
print('Domain chart saved as risks_by_domain.png')

In [ ]:
# ── Mitigation Status Summary ───────────────────────────────────────────────

status_counts = df['Status'].value_counts()
status_colors = ['#639922', '#EF9F27', '#E24B4A']

fig, ax = plt.subplots(figsize=(6, 6))
wedges, texts, autotexts = ax.pie(
    status_counts.values,
    labels=status_counts.index,
    autopct='%1.0f%%',
    colors=status_colors[:len(status_counts)],
    startangle=140,
    wedgeprops=dict(edgecolor='white', linewidth=2)
)
for text in texts: text.set_fontsize(11)
for autotext in autotexts:
    autotext.set_fontsize(12)
    autotext.set_fontweight('bold')

ax.set_title('Mitigation Status', fontsize=13, fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('mitigation_status.png', dpi=150, bbox_inches='tight')
plt.show()
print('Status chart saved as mitigation_status.png')

In [ ]:
# ── Residual Risk Score Calculation ─────────────────────────────────────────

status_multiplier = {'Mitigated': 0.4, 'In Progress': 0.7, 'Planned': 1.0}

df['Adjustment Factor'] = df['Status'].map(status_multiplier)
df['Residual Score'] = (df['Risk Score'] * df['Adjustment Factor']).round(1)

total_inherent = df['Risk Score'].sum()
total_residual = df['Residual Score'].sum().round(1)
reduction_pct = round((1 - total_residual / total_inherent) * 100, 1)

print('=' * 45)
print('RESIDUAL RISK SUMMARY')
print('=' * 45)
print(f'Total inherent risk score : {total_inherent}')
print(f'Total residual risk score : {total_residual}')
print(f'Risk reduction achieved   : {reduction_pct}%')
print('=' * 45)
print('\nTop 5 residual risks (post-mitigation):')
print(df[['Risk ID','Risk Description','Risk Score','Status','Residual Score']]
      .sort_values('Residual Score', ascending=False)
      .head(5)
      .to_string(index=False))

## Summary

This notebook produces four outputs:
- `risk_matrix.png` — 5×5 likelihood-impact heat map with all 18 risks plotted
- `risk_level_distribution.png` — bar chart of risk counts by level
- `risks_by_domain.png` — horizontal bar chart of risks by governance domain
- `mitigation_status.png` — pie chart of current mitigation status

All charts are ready for inclusion in governance reports, board presentations, or client deliverables.

---
*FairFacts Consulting LLC — [fairfactsconsulting.com](https://www.fairfactsconsulting.com)*